Training builds a model. Evaluation tells you whether the model is trustworthy.

Many beginners stop at:

"Accuracy = 92%"

But real ML asks:

92% on what? Which mistakes? Which class fails? Is the model safe to use?

This section teaches that mindset.

Accuracy

Use when:

classes balanced.

Example:

cats vs dogs.

------------------

Precision

Use when:

false positives expensive.

Examples:

medical diagnosis
legal accusations
spam folder

Question:

Can we trust positive predictions?

---------------------

Recall

Use when:

missing positives catastrophic.

Examples:

fraud
disease
security attacks

Question:

Did we catch everything important?

--------------

F1

Use when:

both matter.

Common in:

NLP
sentiment
intent detection
information retrieval

Because:

you want:

reliable predictions
few missed cases

together.

Mental Summary

Confusion Matrix

        ↓

TP FP FN TN

        ↓

Accuracy

overall correctness



Precision

trust positive predictions

Recall

catch all positives

F1

balance precision + recall

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from datasets import load_dataset
import numpy as np
import torch

MODEL_PATH = './distilbert-sst2-final'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_PATH)
model      = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
model.eval()

# Load val set
dataset   = load_dataset('glue', 'sst2')
val_data  = dataset['validation']

# Get all predictions at once using pipeline
clf = pipeline('text-classification', model=model,
               tokenizer=tokenizer, device=-1, return_all_scores=True)

sentences = val_data['sentence']
labels    = val_data['label']

# Run inference in batches
results = clf(sentences, batch_size=64, truncation=True, max_length=128)

# Extract predicted class and confidence
preds  = [np.argmax([r['score'] for r in res]) for res in results]
confs  = [max(r['score'] for r in res) for res in results]

print(f"Evaluated {len(preds)} examples")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

print(classification_report(labels, preds,
      target_names=['negative', 'positive']))

In [ ]:
cm = confusion_matrix(labels, preds)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['neg','pos'],
            yticklabels=['neg','pos'], ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('DistilBERT SST-2 — Confusion Matrix')
plt.tight_layout(); plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"TP={tp}  TN={tn}  FP={fp}  FN={fn}")
print(f"False positive rate: {fp/(fp+tn):.3f}  (ham flagged as spam)")
print(f"False negative rate: {fn/(fn+tp):.3f}  (spam missed)")


Big Picture Flow

What this notebook is doing:

Load saved model
        ↓
Run predictions on full validation set
        ↓
Measure confidence
        ↓
Classification report
        ↓
Confusion matrix
        ↓
Inspect mistakes

This is:

model audit.

---------

Why This Section Matters

Earlier:

you trained models.

Now:

you inspect failure modes.

That is a major shift.

Real ML work is rarely:

"Train and done."

Usually:

Train
→ evaluate
→ inspect failures
→ understand bias
→ improve

This section teaches:

diagnostic thinking.

Pipeline predictions

        ↓

preds + confidence

        ↓

classification report

(per-class metrics)

        ↓

confusion matrix

(mistake map)

        ↓

TP FP FN TN

        ↓

false positive / negative rates

        ↓

understand model bias

Core takeaway:

Accuracy gives a score.

Confusion matrices and error rates explain the score.

In [ ]:
import pandas as pd

results_df = pd.DataFrame({
    'sentence':   sentences,
    'true_label': labels,
    'pred_label': preds,
    'confidence': confs,
    'correct':    [int(p==l) for p,l in zip(preds, labels)]
})

# False positives — predicted positive, actually negative
fp_examples = results_df[
    (results_df['pred_label']==1) & (results_df['true_label']==0)
].sort_values('confidence', ascending=False)

print("=== HIGH-CONFIDENCE FALSE POSITIVES ===")
print("(Model said POSITIVE but answer is NEGATIVE)\n")
for _, row in fp_examples.head(10).iterrows():
    print(f"[conf {row['confidence']:.1%}] {row['sentence'][:80]}")

print()

# False negatives — predicted negative, actually positive
fn_examples = results_df[
    (results_df['pred_label']==0) & (results_df['true_label']==1)
].sort_values('confidence', ascending=False)

print("=== HIGH-CONFIDENCE FALSE NEGATIVES ===")
print("(Model said NEGATIVE but answer is POSITIVE)\n")
for _, row in fn_examples.head(10).iterrows():
    print(f"[conf {row['confidence']:.1%}] {row['sentence'][:80]}")

In [ ]:
from transformers import TrainingArguments, Trainer
import evaluate, numpy as np

accuracy_metric = evaluate.load('accuracy')
f1_metric       = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_metric.compute(predictions=preds, references=labels)['accuracy'],
        'f1':       f1_metric.compute(predictions=preds, references=labels, average='binary')['f1'],
    }

training_args = TrainingArguments(
    output_dir='./distilbert-lora',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=3e-4,          # LoRA can use higher lr than full fine-tuning
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=50,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()


In [ ]:
results = trainer.evaluate()
print("\n=== LoRA Results ===")
print(results)

print("\n=== Comparison ===")
print(f"Full fine-tune (Day 3): ~91% accuracy, ~66M trainable params")
print(f"LoRA (today):           {results['eval_accuracy']*100:.1f}% accuracy, ~300K trainable params")

The real takeaway

This section teaches an important industry habit:

Beginners stop at:

"My model got 92%."

ML engineers continue with:

Where does it fail?

Which examples break it?

Is there bias?

What patterns exist?

Is this a model issue or data issue?

That mindset is what turns model training into model understanding.

One-line summary

Metrics measure performance; failure analysis explains performance.

In [ ]:
# # Option 1: save only the adapter weights (~2MB)
# # The base model is not saved — loaded separately at inference
# model.save_pretrained('./lora-adapter-only')

# # This saves only the tiny adapter files:
# # adapter_config.json + adapter_model.safetensors

In [ ]:

# # Option 2: merge adapters into base model then save
# # Result: a normal model file with LoRA baked in — no peft needed at inference
# from peft import PeftModel

# # Load base + adapter and merge
# base   = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
# merged = PeftModel.from_pretrained(base, './lora-adapter-only')
# merged = merged.merge_and_unload()     # merges A×B into W, removes LoRA structure

# merged.save_pretrained('./distilbert-lora-merged')
# tokenizer.save_pretrained('./distilbert-lora-merged')
# print("Merged model saved")

In [ ]:
# from transformers import pipeline

# clf = pipeline(
#     'text-classification',
#     model='./distilbert-lora-merged',
#     tokenizer='./distilbert-lora-merged',
#     device=-1
# )

# tests = [
#     "An absolute masterpiece from start to finish.",
#     "Painfully dull and a complete waste of potential.",
#     "Not the worst thing I've seen but close.",
#     "Genuinely surprised by how good this was.",
# ]

# for res, text in zip(clf(tests), tests):
#     print(f"[{res['label']:8s} {res['score']:.1%}]  {text}")

In [ ]:
# import os

# def folder_size(path):
#     total = sum(os.path.getsize(os.path.join(p,f))
#                 for p,_,files in os.walk(path) for f in files)
#     return total / 1e6  # MB

# print(f"Adapter only:    {folder_size('./lora-adapter-only'):.1f} MB")
# print(f"Merged model:    {folder_size('./distilbert-lora-merged'):.1f} MB")
# print(f"Full fine-tune:  {folder_size('./distilbert-sst2-final'):.1f} MB")

<!-- option - 1

Layman analogy

Imagine:

DistilBERT

PlayStation console.

LoRA adapter

Game disc.

Saving adapter only means:

You save just the game.

Not the entire PlayStation.

Later you insert the game into the console again.

Small adapter + existing base model

-------

option - 2

Sometimes you don't want:

base + adapter

You want:

one normal model

This is merging. 

What this does

Loads a fresh base DistilBERT.

No adapters yet.

Just original weights.

Think:

load the PlayStation 
Think:

insert game disc into console

Adapter-only vs merged

This is the main comparison.

| Feature          | Adapter Only | Merged Model |
| ---------------- | ------------ | ------------ |
| Size             | Tiny         | Large        |
| Needs base model | Yes          | No           |
| Needs PEFT       | Yes          | No           |
| Flexible         | Very         | Less         |
| Easy deployment  | Moderate     | Easy         | -->


<!-- When to use adapter-only

Best when:

many tasks
shared base model
storage matters
research workflows

Example:

Company has:

sentiment adapter,
legal adapter,
medical adapter

All reuse one base model.

Efficient.

When to use merged

Best when:

deploying app/API,
simple inference,
sharing model,
production endpoint

Example:

You want:

pipeline(...)

and done.

Merged is simpler. -->

<!-- Big-picture intuition

Think of LoRA like phone filters.

Base model

Phone camera app.

Adapter

Instagram filter.

You have two choices:

Option 1

Save filter only.

Small.

Reuse on many phones.

(Adapter-only)

Option 2

Bake filter permanently into photo.

One final image.

(Merged model)

Both work.

Choice depends on deployment needs. -->